# Core Colab: Data Augmentation and Generalization in TensorFlow and PyTorch

This notebook is written for beginners.  
It shows small **A/B tests** for generalization techniques using simple image classification tasks.

Covered topics:
1. L1 and L2 regularization  
2. Dropout  
3. Early stopping  
4. Monte Carlo Dropout  
5. Weight initialization and when to use what  
6. Batch normalization  
7. Custom dropout and custom regularization  
8. Callbacks and TensorBoard  
9. Keras Tuner  
10. KerasCV data augmentation  
11. A small PyTorch version of the same ideas

In [ ]:
# If you are running this in Google Colab, uncomment this cell.
# It installs the extra libraries used in this notebook.

# !pip -q install keras-tuner keras-cv tensorflow-datasets

In [ ]:
import os
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def set_tf_seed(seed=42):
    import tensorflow as tf
    tf.keras.utils.set_random_seed(seed)

def set_torch_seed(seed=42):
    import torch
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## Part 1: TensorFlow setup
We will use **Fashion MNIST** because it is small and beginner friendly.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks

set_tf_seed(SEED)

(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Add channel dimension
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

# Train / validation split
x_val = x_train[-10000:]
y_val = y_train[-10000:]
x_train_small = x_train[:-10000]
y_train_small = y_train[:-10000]

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

print("Train:", x_train_small.shape, y_train_small.shape)
print("Val:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)

In [ ]:
def show_examples(images, labels, n=10):
    plt.figure(figsize=(12, 3))
    for i in range(n):
        plt.subplot(2, 5, i + 1)
        plt.imshow(images[i].squeeze(), cmap="gray")
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
    plt.tight_layout()
    plt.show()

show_examples(x_train_small, y_train_small)

## A reusable training function
We will use the same helper for all TensorFlow A/B tests.

In [ ]:
def build_tf_model(
    l1_value=0.0,
    l2_value=0.0,
    dropout_rate=0.0,
    use_batchnorm=False,
    kernel_initializer="glorot_uniform",
    use_custom_dropout=False
):
    reg = regularizers.L1L2(l1=l1_value, l2=l2_value)

    inputs = keras.Input(shape=(28, 28, 1))
    x = layers.Conv2D(
        32, 3, padding="same",
        kernel_initializer=kernel_initializer,
        kernel_regularizer=reg
    )(inputs)
    if use_batchnorm:
        x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(
        64, 3, padding="same",
        kernel_initializer=kernel_initializer,
        kernel_regularizer=reg
    )(x)
    if use_batchnorm:
        x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Flatten()(x)
    x = layers.Dense(
        128,
        kernel_initializer=kernel_initializer,
        kernel_regularizer=reg
    )(x)
    if use_batchnorm:
        x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    if use_custom_dropout:
        x = CustomDropout(drop_rate=dropout_rate)(x)
    else:
        x = layers.Dropout(dropout_rate)(x)

    outputs = layers.Dense(10, activation="softmax")(x)
    model = keras.Model(inputs, outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


def train_tf_model(model, name, epochs=5, use_earlystop=False, use_tensorboard=False):
    cb = []
    if use_earlystop:
        cb.append(callbacks.EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True
        ))
    if use_tensorboard:
        log_dir = os.path.join("logs", name)
        cb.append(callbacks.TensorBoard(log_dir=log_dir))

    history = model.fit(
        x_train_small, y_train_small,
        validation_data=(x_val, y_val),
        epochs=epochs,
        batch_size=128,
        verbose=0,
        callbacks=cb
    )

    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"{name}: test_acc={test_acc:.4f}, test_loss={test_loss:.4f}")
    return history, test_acc, test_loss

## A/B Test 1: Baseline vs L1/L2 regularization

- **L1** pushes some weights toward exactly zero.  
- **L2** keeps weights small and smooth.  
- In practice:
  - Use **L2** most of the time as a safe default.
  - Use **L1** when you want sparsity.

In [ ]:
baseline_tf = build_tf_model()
hist_baseline_tf, acc_baseline_tf, _ = train_tf_model(baseline_tf, "tf_baseline", epochs=5)

l2_tf = build_tf_model(l2_value=1e-4)
hist_l2_tf, acc_l2_tf, _ = train_tf_model(l2_tf, "tf_l2", epochs=5)

l1_tf = build_tf_model(l1_value=1e-5)
hist_l1_tf, acc_l1_tf, _ = train_tf_model(l1_tf, "tf_l1", epochs=5)

results_tf = [
    ["Baseline", acc_baseline_tf],
    ["L2", acc_l2_tf],
    ["L1", acc_l1_tf],
]
pd.DataFrame(results_tf, columns=["Experiment", "Test Accuracy"])

## A/B Test 2: Baseline vs Dropout

Dropout randomly drops activations during training, which helps prevent overfitting.

In [ ]:
dropout_tf = build_tf_model(dropout_rate=0.3)
hist_dropout_tf, acc_dropout_tf, _ = train_tf_model(dropout_tf, "tf_dropout", epochs=5)

pd.DataFrame(
    [["Baseline", acc_baseline_tf], ["Dropout=0.3", acc_dropout_tf]],
    columns=["Experiment", "Test Accuracy"]
)

## A/B Test 3: Early stopping

Early stopping stops training when validation performance stops improving.
It is a very common and practical regularization method.

In [ ]:
earlystop_tf = build_tf_model(dropout_rate=0.2)
hist_early_tf, acc_early_tf, _ = train_tf_model(
    earlystop_tf,
    "tf_earlystop",
    epochs=15,
    use_earlystop=True
)

print("Epochs actually used:", len(hist_early_tf.history["loss"]))

## A/B Test 4: Batch normalization

Batch normalization often stabilizes training and can make deeper networks easier to train.

In [ ]:
bn_tf = build_tf_model(use_batchnorm=True)
hist_bn_tf, acc_bn_tf, _ = train_tf_model(bn_tf, "tf_batchnorm", epochs=5)

pd.DataFrame(
    [["Baseline", acc_baseline_tf], ["BatchNorm", acc_bn_tf]],
    columns=["Experiment", "Test Accuracy"]
)

## A/B Test 5: Weight initialization

Common choices:
- **glorot_uniform / Xavier**: good default for tanh or general dense networks
- **he_normal / he_uniform**: best choice when using ReLU
- **lecun_normal**: often used with SELU

Because our network uses **ReLU**, `he_normal` is usually a good choice.

In [ ]:
init_results = []

for init_name in ["glorot_uniform", "he_normal", "lecun_normal"]:
    model = build_tf_model(kernel_initializer=init_name)
    _, acc, _ = train_tf_model(model, f"tf_init_{init_name}", epochs=5)
    init_results.append([init_name, acc])

pd.DataFrame(init_results, columns=["Initializer", "Test Accuracy"])

## Custom dropout and custom regularization
Below we create:
1. A custom dropout layer
2. A custom activity regularizer

In [ ]:
class CustomDropout(layers.Layer):
    def __init__(self, drop_rate=0.2, **kwargs):
        super().__init__(**kwargs)
        self.drop_rate = drop_rate

    def call(self, inputs, training=None):
        if training:
            mask = tf.cast(
                tf.random.uniform(tf.shape(inputs)) >= self.drop_rate,
                inputs.dtype
            )
            return inputs * mask / (1.0 - self.drop_rate)
        return inputs


class SimpleActivityRegularizer(layers.Layer):
    def __init__(self, penalty=1e-4, **kwargs):
        super().__init__(**kwargs)
        self.penalty = penalty

    def call(self, inputs):
        self.add_loss(self.penalty * tf.reduce_mean(tf.abs(inputs)))
        return inputs

In [ ]:
def build_tf_model_with_custom_regularizer():
    inputs = keras.Input(shape=(28, 28, 1))
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = SimpleActivityRegularizer(penalty=1e-4)(x)
    x = CustomDropout(drop_rate=0.3)(x)
    outputs = layers.Dense(10, activation="softmax")(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

custom_tf = build_tf_model_with_custom_regularizer()
hist_custom_tf, acc_custom_tf, _ = train_tf_model(custom_tf, "tf_custom", epochs=5)
print("Custom regularization + custom dropout accuracy:", acc_custom_tf)

## Monte Carlo Dropout

At test time, dropout is usually turned off.  
For **Monte Carlo Dropout**, we keep dropout active at inference time and run multiple forward passes.

This gives:
- an average prediction
- an uncertainty estimate

This is useful when you want the model to say:  
**I am less confident about this sample.**

In [ ]:
mc_model = build_tf_model(dropout_rate=0.4)
_ = train_tf_model(mc_model, "tf_mc_dropout", epochs=5)

def mc_predict(model, x, n_samples=20):
    probs = []
    for _ in range(n_samples):
        # training=True keeps dropout active
        pred = model(x, training=True).numpy()
        probs.append(pred)
    probs = np.stack(probs, axis=0)
    mean_probs = probs.mean(axis=0)
    std_probs = probs.std(axis=0)
    return mean_probs, std_probs

sample_images = x_test[:5]
mean_probs, std_probs = mc_predict(mc_model, sample_images, n_samples=20)

for i in range(5):
    pred_class = np.argmax(mean_probs[i])
    uncertainty = std_probs[i].mean()
    print(f"Sample {i}: predicted={class_names[pred_class]}, uncertainty={uncertainty:.4f}, true={class_names[y_test[i]]}")

## Callbacks and TensorBoard
We already used **EarlyStopping**.  
Now we will also use **TensorBoard**.

In [ ]:
tb_model = build_tf_model(dropout_rate=0.2, use_batchnorm=True)
_ = train_tf_model(
    tb_model,
    "tf_tensorboard_demo",
    epochs=5,
    use_earlystop=True,
    use_tensorboard=True
)

print("TensorBoard logs were written to the logs/ folder.")
print("In Colab, you can run:")
print("%load_ext tensorboard")
print("%tensorboard --logdir logs")

## Keras Tuner
Keras Tuner helps us search for good hyperparameters automatically.
Here we tune:
- dropout rate
- dense layer width
- learning rate

In [ ]:
import keras_tuner as kt

def build_tuner_model(hp):
    dropout_rate = hp.Choice("dropout_rate", values=[0.1, 0.2, 0.3])
    units = hp.Choice("units", values=[64, 128, 256])
    lr = hp.Choice("learning_rate", values=[1e-2, 1e-3])

    inputs = keras.Input(shape=(28, 28, 1))
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(units, activation="relu")(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(10, activation="softmax")(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

tuner = kt.RandomSearch(
    build_tuner_model,
    objective="val_accuracy",
    max_trials=3,
    overwrite=True,
    directory="kt_dir",
    project_name="fashion_mnist_demo"
)

tuner.search(
    x_train_small, y_train_small,
    validation_data=(x_val, y_val),
    epochs=3,
    batch_size=128,
    verbose=0
)

best_hp = tuner.get_best_hyperparameters(1)[0]
print("Best dropout_rate:", best_hp.get("dropout_rate"))
print("Best units:", best_hp.get("units"))
print("Best learning_rate:", best_hp.get("learning_rate"))

## KerasCV data augmentation
We will apply image augmentations before training.

In [ ]:
import keras_cv

augmenter = keras.Sequential([
    keras_cv.layers.RandomFlip(mode="horizontal"),
    keras_cv.layers.RandomRotation(factor=0.1),
    keras_cv.layers.RandomCutout(height_factor=0.1, width_factor=0.1),
])

augmented_images = augmenter(x_train_small[:8])

plt.figure(figsize=(10, 4))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(augmented_images[i].numpy().squeeze(), cmap="gray")
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def build_tf_augmented_model():
    inputs = keras.Input(shape=(28, 28, 1))
    x = augmenter(inputs)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(10, activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

aug_tf = build_tf_augmented_model()
_, acc_aug_tf, _ = train_tf_model(aug_tf, "tf_keras_cv_aug", epochs=5)

pd.DataFrame(
    [["Baseline", acc_baseline_tf], ["KerasCV augmentation", acc_aug_tf]],
    columns=["Experiment", "Test Accuracy"]
)

## Part 2: PyTorch setup
Now we repeat the same ideas in PyTorch.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

set_torch_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

transform = transforms.ToTensor()

train_dataset = datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)

train_len = len(train_dataset) - 10000
val_len = 10000
train_dataset, val_dataset = random_split(train_dataset, [train_len, val_len])

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [ ]:
class CustomTorchDropout(nn.Module):
    def __init__(self, p=0.2):
        super().__init__()
        self.p = p

    def forward(self, x):
        if self.training:
            mask = (torch.rand_like(x) > self.p).float()
            return x * mask / (1.0 - self.p)
        return x


class TorchCNN(nn.Module):
    def __init__(
        self,
        dropout_p=0.0,
        use_batchnorm=False,
        init_name="xavier",
        use_custom_dropout=False
    ):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32) if use_batchnorm else nn.Identity()
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64) if use_batchnorm else nn.Identity()

        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.bn3 = nn.BatchNorm1d(128) if use_batchnorm else nn.Identity()

        self.dropout = CustomTorchDropout(dropout_p) if use_custom_dropout else nn.Dropout(dropout_p)
        self.fc2 = nn.Linear(128, 10)

        self.init_weights(init_name)

    def init_weights(self, init_name):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                if init_name == "xavier":
                    nn.init.xavier_uniform_(m.weight)
                elif init_name == "he":
                    nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                elif init_name == "normal":
                    nn.init.normal_(m.weight, mean=0.0, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = torch.relu(self.bn1(self.conv1(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.relu(self.bn2(self.conv2(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.flatten(x, 1)
        x = torch.relu(self.bn3(self.fc1(x)))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [ ]:
def evaluate_torch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)

            total_loss += loss.item() * x.size(0)
            total_correct += (logits.argmax(dim=1) == y).sum().item()
            total_count += x.size(0)

    return total_loss / total_count, total_correct / total_count


def train_torch_model(
    name,
    dropout_p=0.0,
    weight_decay=0.0,
    l1_lambda=0.0,
    use_batchnorm=False,
    init_name="xavier",
    use_custom_dropout=False,
    epochs=5,
    early_stop=False
):
    model = TorchCNN(
        dropout_p=dropout_p,
        use_batchnorm=use_batchnorm,
        init_name=init_name,
        use_custom_dropout=use_custom_dropout
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=weight_decay)

    best_val_loss = float("inf")
    best_state = None
    patience = 2
    wait = 0

    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)

            if l1_lambda > 0:
                l1_penalty = sum(p.abs().sum() for p in model.parameters())
                loss = loss + l1_lambda * l1_penalty

            loss.backward()
            optimizer.step()

        val_loss, val_acc = evaluate_torch(model, val_loader, criterion)

        if early_stop:
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    print(f"{name}: early stopping at epoch {epoch+1}")
                    break

    if best_state is not None:
        model.load_state_dict(best_state)

    test_loss, test_acc = evaluate_torch(model, test_loader, criterion)
    print(f"{name}: test_acc={test_acc:.4f}, test_loss={test_loss:.4f}")
    return model, test_acc

## PyTorch A/B tests

In [ ]:
torch_results = []

torch_baseline_model, torch_baseline_acc = train_torch_model("torch_baseline", epochs=5)
torch_results.append(["Baseline", torch_baseline_acc])

_, torch_l2_acc = train_torch_model("torch_l2", weight_decay=1e-4, epochs=5)
torch_results.append(["L2 weight decay", torch_l2_acc])

_, torch_l1_acc = train_torch_model("torch_l1", l1_lambda=1e-7, epochs=5)
torch_results.append(["L1", torch_l1_acc])

_, torch_dropout_acc = train_torch_model("torch_dropout", dropout_p=0.3, epochs=5)
torch_results.append(["Dropout", torch_dropout_acc])

_, torch_bn_acc = train_torch_model("torch_batchnorm", use_batchnorm=True, epochs=5)
torch_results.append(["BatchNorm", torch_bn_acc])

_, torch_early_acc = train_torch_model("torch_earlystop", dropout_p=0.2, epochs=12, early_stop=True)
torch_results.append(["EarlyStopping", torch_early_acc])

_, torch_custom_acc = train_torch_model("torch_custom_dropout", dropout_p=0.3, use_custom_dropout=True, epochs=5)
torch_results.append(["CustomDropout", torch_custom_acc])

for init_name in ["xavier", "he", "normal"]:
    _, init_acc = train_torch_model(f"torch_init_{init_name}", init_name=init_name, epochs=5)
    torch_results.append([f"Init: {init_name}", init_acc])

pd.DataFrame(torch_results, columns=["Experiment", "Test Accuracy"])

## Monte Carlo Dropout in PyTorch
We switch the model to `train()` mode during inference so dropout stays active.

In [ ]:
mc_torch_model, _ = train_torch_model("torch_mc_dropout", dropout_p=0.4, epochs=5)

def mc_predict_torch(model, x, n_samples=20):
    model.train()  # keep dropout on
    probs = []
    with torch.no_grad():
        for _ in range(n_samples):
            logits = model(x)
            probs.append(torch.softmax(logits, dim=1).cpu().numpy())
    probs = np.stack(probs, axis=0)
    return probs.mean(axis=0), probs.std(axis=0)

sample_batch, sample_labels = next(iter(test_loader))
sample_batch = sample_batch[:5].to(device)

mean_probs_t, std_probs_t = mc_predict_torch(mc_torch_model, sample_batch, n_samples=20)

for i in range(5):
    pred_class = int(np.argmax(mean_probs_t[i]))
    uncertainty = float(std_probs_t[i].mean())
    print(f"Sample {i}: predicted={class_names[pred_class]}, uncertainty={uncertainty:.4f}")

## Final summary
This notebook showed beginner-friendly A/B tests for:
- L1 / L2
- Dropout
- Early stopping
- Monte Carlo Dropout
- Initializations
- Batch normalization
- Custom dropout and regularization
- Callbacks and TensorBoard
- Keras Tuner
- KerasCV augmentation
- TensorFlow and PyTorch implementations